# Neural Theorem Prover — Interactive Demo

This notebook demonstrates the end-to-end pipeline:
1. Load the pretrained ReProver tactic model
2. Run proof search on example theorems
3. Verify the found proofs with Lean 4

**Prerequisites** (see README.md):
- `bash setup.sh` completed
- `export GITHUB_ACCESS_TOKEN=<your_token>`
- `cd lean_project && lake build` finished
- Kernel set to the `.venv` environment

In [ ]:
import os
import sys
import logging

# Add repo root to path
sys.path.insert(0, os.path.abspath('..'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# Check environment
if not os.environ.get('GITHUB_ACCESS_TOKEN'):
    print('WARNING: GITHUB_ACCESS_TOKEN not set. LeanDojo will fail.')
    print('Set it with: export GITHUB_ACCESS_TOKEN=<your_token>')
else:
    print('GITHUB_ACCESS_TOKEN is set — OK')

## 1. Initialize the Proof Search System

In [ ]:
from prover import ProofSearch

prover = ProofSearch(
    model_path='../models/pretrained/leandojo-lean4-tacgen-byt5-small',
    lean_project='../lean_project',
    top_k=32,
)

print('ProofSearch initialized')

## 2. Prove a Simple Theorem

In [ ]:
result = prover.prove(
    theorem='∀ n : ℕ, n + 0 = n',
    hypotheses=[],
    timeout=60,
    max_depth=20,
)

print(f'Verified: {result.verified}')
print(f'Elapsed: {result.elapsed_seconds:.2f}s')
print(f'Nodes expanded: {result.search_nodes_expanded}')
print()
if result.proof:
    print('Proof:')
    print(result.proof)
    print()
    print('Steps:')
    for state, tactic in result.steps:
        print(f'  State: {state[:80]}...')
        print(f'  Tactic: {tactic}')
        print()
else:
    print('No proof found.')
    if result.error:
        print(f'Error: {result.error}')

## 3. Batch Test on All Baseline Theorems

In [ ]:
test_theorems = [
    ('n + 0 = n',                  ['n : ℕ']),
    ('0 + n = n',                  ['n : ℕ']),
    ('n + m = m + n',              ['n m : ℕ']),
    ('(n + m) + k = n + (m + k)',  ['n m k : ℕ']),
    ('P ∧ Q → Q ∧ P',             ['P Q : Prop']),
    ('P → P',                      ['P : Prop']),
]

results = prover.batch_prove(test_theorems, timeout=60, max_depth=20)

print('\n=== Summary ===')
for (thm, _), r in zip(test_theorems, results):
    status = '✓' if r.verified else '✗'
    print(f'{status} [{r.elapsed_seconds:.1f}s] {thm}')
    if r.proof:
        print(f'    Proof: {r.proof[:60]}')

n_success = sum(r.verified for r in results)
print(f'\n{n_success}/{len(results)} theorems proved')

## 4. Extract Training Data (optional)

Run data extraction on a small mathlib module to see the output format.

In [ ]:
# This cell is slow (LeanDojo traces the full module).
# Uncomment to run.

# import sys
# sys.path.insert(0, '..')
# from data.extract import extract_module, print_summary, save_jsonl
#
# records = extract_module(
#     module_name='Mathlib.Data.Nat.Basic',
#     lean_project_path='../lean_project',
#     max_theorems=20,  # limit for demo
# )
#
# print_summary(records, 'Mathlib.Data.Nat.Basic')
#
# if records:
#     print('First record:')
#     import json
#     from dataclasses import asdict
#     print(json.dumps(asdict(records[0]), indent=2, ensure_ascii=False)[:1000])